In [1]:
import pandas as pd

# -----------------------------
# Path to CSV file
# -----------------------------
csv_path = r"C:\Users\esafrygina\QCS_eraser\TCGA_OV_clinical.csv"

# -----------------------------
# Load CSV into pandas
# -----------------------------
clinical_df = pd.read_csv(csv_path)

# -----------------------------
# Display first 5 rows
# -----------------------------
print("\nClinical data preview:")
print(clinical_df.head())

# -----------------------------
# Display column names
# -----------------------------
print("\nClinical data columns:")
for col in clinical_df.columns:
    print(col)

# -----------------------------
# Optional: show dataframe shape
# -----------------------------
print("\nData shape:")
print(clinical_df.shape)


Clinical data preview:
   project  submitter_id  figo_stage synchronous_malignancy  \
0  TCGA-OV  TCGA-04-1335    Stage IB           Not Reported   
1  TCGA-OV  TCGA-04-1360    Stage IA           Not Reported   
2  TCGA-OV  TCGA-23-2649  Stage IIIC           Not Reported   
3  TCGA-OV  TCGA-42-2593  Stage IIIC           Not Reported   
4  TCGA-OV  TCGA-24-2261  Stage IIIC           Not Reported   

   days_to_diagnosis laterality  created_datetime  last_known_disease_status  \
0                0.0  Bilateral               NaN                        NaN   
1                0.0        NaN               NaN                        NaN   
2                0.0       Left               NaN                        NaN   
3                0.0  Bilateral               NaN                        NaN   
4                0.0       Left               NaN                        NaN   

  tissue_or_organ_of_origin  days_to_last_follow_up  ...  \
0                     Ovary                    55.0  ...

In [2]:
import pandas as pd
import numpy as np

# ---------------------------------------------------
# Load TCGA-OV clinical CSV
# ---------------------------------------------------
csv_path = r"C:\Users\esafrygina\QCS_eraser\TCGA_OV_clinical.csv"

clinical_tcga_ov = pd.read_csv(csv_path)

# ---------------------------------------------------
# Remove "Not Reported" vital status entries
# ---------------------------------------------------
clinical_tcga_ov = clinical_tcga_ov[
    clinical_tcga_ov["vital_status"].isin(["Alive", "Dead"])
].copy()

# ---------------------------------------------------
# Create deceased column
# TRUE if Dead, FALSE if Alive
# ---------------------------------------------------
clinical_tcga_ov["deceased"] = (
    clinical_tcga_ov["vital_status"] == "Dead"
)

# ---------------------------------------------------
# Create overall survival column
# If Alive  -> days_to_last_follow_up
# If Dead   -> days_to_death
# ---------------------------------------------------
clinical_tcga_ov["overall_survival"] = np.where(
    clinical_tcga_ov["vital_status"] == "Alive",
    clinical_tcga_ov["days_to_last_follow_up"],
    clinical_tcga_ov["days_to_death"]
)

# ---------------------------------------------------
# Print counts table
# ---------------------------------------------------
table_df = clinical_tcga_ov["vital_status"].value_counts()

print("\nAlive / Dead counts:")
print(table_df)

# ---------------------------------------------------
# Preview relevant columns
# ---------------------------------------------------
print("\nClinical data preview:")
print(
    clinical_tcga_ov[
        [
            "bcr_patient_barcode",
            "vital_status",
            "deceased",
            "overall_survival"
        ]
    ].head()
)



Alive / Dead counts:
vital_status
Dead     349
Alive    236
Name: count, dtype: int64

Clinical data preview:
  bcr_patient_barcode vital_status  deceased  overall_survival
0        TCGA-04-1335         Dead      True              55.0
1        TCGA-04-1360        Alive     False               NaN
2        TCGA-23-2649        Alive     False             116.0
3        TCGA-42-2593         Dead      True              45.0
4        TCGA-24-2261         Dead      True              24.0


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# =====================================================
# LOAD CLINICAL DATA
# =====================================================

csv_path = r"C:\Users\esafrygina\QCS_eraser\TCGA_OV_clinical.csv"

clinical = pd.read_csv(csv_path)

# keep only alive/dead
clinical = clinical[
    clinical["vital_status"].isin(["Alive", "Dead"])
].copy()

# event indicator
clinical["event"] = (
    clinical["vital_status"] == "Dead"
).astype(int)

# survival time
clinical["time"] = np.where(
    clinical["event"] == 1,
    clinical["days_to_death"],
    clinical["days_to_last_follow_up"]
)

# remove missing
clinical = clinical.dropna(subset=["time"])

# =====================================================
# DUMMY PATCH FEATURES
# =====================================================

# number of patients
N = len(clinical)

# suppose each slide has:
n_patches = 50

# feature dimension
d = 10

# random FM-like embeddings
# shape:
# [patients, patches, features]

X = torch.randn(N, n_patches, d)

# =====================================================
# SURVIVAL TENSORS
# =====================================================

times = torch.tensor(
    clinical["time"].values,
    dtype=torch.float32
)

events = torch.tensor(
    clinical["event"].values,
    dtype=torch.float32
)

# =====================================================
# SORT BY SURVIVAL TIME DESCENDING
# IMPORTANT FOR COX LOSS
# =====================================================

order = torch.argsort(times, descending=True)

X = X[order]
times = times[order]
events = events[order]

# =====================================================
# ABMIL MODEL
# =====================================================

class ABMILSurvival(nn.Module):

    def __init__(self, in_dim=10, hidden_dim=32):

        super().__init__()

        # attention network
        self.attention_V = nn.Linear(in_dim, hidden_dim)
        self.attention_w = nn.Linear(hidden_dim, 1)

        # latent projection
        self.projector = nn.Linear(in_dim, 16)

        # risk head
        self.risk_head = nn.Linear(16, 1)

    def forward(self, x):

        # x shape:
        # [B, patches, features]

        A = torch.tanh(
            self.attention_V(x)
        )

        A = self.attention_w(A)

        # attention weights
        A = torch.softmax(A, dim=1)

        # attention pooling
        M = torch.sum(A * x, dim=1)

        # latent embedding
        H = self.projector(M)

        # risk score
        risk = self.risk_head(H)

        return risk.squeeze(), H, A


# =====================================================
# COX LOSS
# =====================================================

def cox_loss(risk, events):

    hazard = torch.exp(risk)

    log_cumsum_hazard = torch.log(
        torch.cumsum(hazard, dim=0)
    )

    uncensored_likelihood = (
        risk - log_cumsum_hazard
    ) * events

    neg_log_likelihood = -torch.sum(
        uncensored_likelihood
    ) / torch.sum(events)

    return neg_log_likelihood

# =====================================================
# TRAINING
# =====================================================

model = ABMILSurvival()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)

for epoch in range(50):

    optimizer.zero_grad()

    risk, latent, attention = model(X)

    loss = cox_loss(risk, events)

    loss.backward()

    optimizer.step()

    print(
        f"Epoch {epoch:03d} | "
        f"Loss: {loss.item():.4f}"
    )

# =====================================================
# OUTPUTS
# =====================================================

print("\nRisk scores shape:")
print(risk.shape)

print("\nLatent embeddings shape:")
print(latent.shape)

print("\nAttention shape:")
print(attention.shape)